# Sigma4 — Versi Pengembangan dari sigma3

Perubahan dari versi sebelumnya:
1. **GroupKFold berdasarkan `source_id`** — supaya validasi lebih jujur (tidak ada kebocoran data antar fold dari sumber yang sama).
2. **Fitur tambahan**: rasio antar kation, ringkasan spectral band, dan kelompok wilayah dari lokasi.
3. **Tambah model CatBoost** sebagai model ketiga.
4. **N_TRIALS dinaikkan** untuk pencarian setelan yang lebih optimal.
5. **Blending 3 model** dengan pencarian bobot otomatis.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.cluster import KMeans
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [2]:
RANDOM_STATE = 42
N_FOLDS = 5
TUNE_FOLDS = 3
N_TRIALS = 30      # dinaikkan dari 12 -> 30, cari setelan lebih optimal (lebih lama tapi lebih akurat)
N_GEO_CLUSTERS = 20 # jumlah kelompok wilayah dari lat/long

In [3]:
DATA_DIR = '/kaggle/input/competitions/seleksi-data-science-academy-compfest-18'
OUT_DIR  = '/kaggle/working/'
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test  = pd.read_csv(f'{DATA_DIR}/test.csv')

In [4]:
TARGET = 'property_organic_content'
ID_COL = 'sample_id'
GROUP_COL = 'source_id'  # dipakai untuk GroupKFold, biar 1 sumber data tidak kepisah antar fold

In [5]:
cat_cols = ['source_id', 'has_band_A_spectrum', 'has_band_B_spectrum',
            'sampling_strategy', 'sampling_depth_cm', 'geo_zone_macro',
            'geo_zone_micro', 'geo_zone_meso', 'land_cover_type',
            'biome', 'parent_rock_type']

## Feature Engineering (ditambah dari versi sigma3)

Tambahan baru:
- **Rasio kation**: perbandingan antar Ca, Mg, Na, dan CEC — kadang perbandingan lebih bermakna daripada angka mentah sendiri-sendiri.
- **Ringkasan spectral**: rata-rata & simpangan dari kolom-kolom PC band A dan band B, supaya mesin punya "gambaran umum" spektrum, bukan cuma detail per komponen.
- **Kelompok wilayah (geo cluster)**: lokasi (lat/long) dikelompokkan pakai KMeans jadi beberapa "wilayah", supaya mesin bisa menangkap pola khas per wilayah tanpa harus hafal koordinat persis.

In [6]:
def feature_engineer(df):
    df = df.copy()

    # flag data hilang (dari versi sigma3)
    df['flag_has_coord'] = df['latitude'].notna().astype(int)
    df['flag_has_acidity'] = df['property_acidity_index'].notna().astype(int)
    df['flag_has_cation_na'] = df['cation_Na'].notna().astype(int)
    df['flag_has_cation_set'] = df['cation_Ca'].notna().astype(int)
    df['flag_has_particle'] = df['property_particle_coarse'].notna().astype(int)
    df['flag_has_band_B'] = (df['has_band_B_spectrum'] == 'YES').astype(int)

    # --- BARU: rasio antar kation ---
    eps = 1e-6
    df['ratio_ca_mg'] = df['cation_Ca'] / (df['cation_Mg'] + eps)
    df['ratio_ca_cec'] = df['cation_Ca'] / (df['cation_exchange_capacity'] + eps)
    df['ratio_mg_cec'] = df['cation_Mg'] / (df['cation_exchange_capacity'] + eps)
    df['ratio_na_cec'] = df['cation_Na'] / (df['cation_exchange_capacity'] + eps)
    df['cation_sum'] = df[['cation_Ca', 'cation_Mg', 'cation_Na']].sum(axis=1)

    # --- BARU: ringkasan kolom spectral ---
    band_a_cols = [c for c in df.columns if c.startswith('spectral_band_A_PC')]
    band_b_cols = [c for c in df.columns if c.startswith('spectral_band_B_PC')]
    df['band_A_mean'] = df[band_a_cols].mean(axis=1)
    df['band_A_std'] = df[band_a_cols].std(axis=1)
    df['band_B_mean'] = df[band_b_cols].mean(axis=1)
    df['band_B_std'] = df[band_b_cols].std(axis=1)

    # partikel tanah: rasio kasar vs halus
    df['ratio_coarse_fine'] = df['property_particle_coarse'] / (df['property_particle_fine'] + eps)

    for c in cat_cols:
        df[c] = df[c].astype('category')
    return df

In [7]:
train_fe = feature_engineer(train)
test_fe = feature_engineer(test)

In [8]:
# --- BARU: kelompok wilayah (geo cluster) pakai KMeans ---
# dilatih hanya dari data yang punya koordinat, baris tanpa koordinat diberi label -1
coord_train_mask = train_fe['latitude'].notna() & train_fe['longitude'].notna()
coord_test_mask = test_fe['latitude'].notna() & test_fe['longitude'].notna()

kmeans = KMeans(n_clusters=N_GEO_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
kmeans.fit(train_fe.loc[coord_train_mask, ['latitude', 'longitude']])

train_fe['geo_cluster'] = -1
test_fe['geo_cluster'] = -1
train_fe.loc[coord_train_mask, 'geo_cluster'] = kmeans.predict(train_fe.loc[coord_train_mask, ['latitude', 'longitude']])
test_fe.loc[coord_test_mask, 'geo_cluster'] = kmeans.predict(test_fe.loc[coord_test_mask, ['latitude', 'longitude']])

train_fe['geo_cluster'] = train_fe['geo_cluster'].astype('category')
test_fe['geo_cluster'] = test_fe['geo_cluster'].astype('category')
cat_cols = cat_cols + ['geo_cluster']

In [9]:
# samakan daftar kategori antara train & test (termasuk geo_cluster yang baru ditambah)
for c in cat_cols:
    cats = pd.api.types.union_categoricals([train_fe[c], test_fe[c]]).categories
    train_fe[c] = train_fe[c].cat.set_categories(cats)
    test_fe[c] = test_fe[c].cat.set_categories(cats)

In [10]:
# versi kategori diubah jadi angka, dipakai untuk XGBoost
train_xgb = train_fe.copy()
test_xgb = test_fe.copy()
for c in cat_cols:
    train_xgb[c] = train_xgb[c].cat.codes.replace(-1, np.nan)
    test_xgb[c] = test_xgb[c].cat.codes.replace(-1, np.nan)

In [11]:
drop_cols = [ID_COL, TARGET]
feature_cols = [c for c in train_fe.columns if c not in drop_cols]
cat_col_idx = [feature_cols.index(c) for c in cat_cols]  # dibutuhkan CatBoost

In [12]:
X = train_fe[feature_cols]
y = np.log1p(train_fe[TARGET])
X_test = test_fe[feature_cols]
groups = train_fe[GROUP_COL]

In [13]:
X_xgb = train_xgb[feature_cols]
X_test_xgb = test_xgb[feature_cols]

In [14]:
# CatBoost bisa pakai kategori as-is asal tipe string (bukan pandas category)
X_cat = X.copy()
X_test_cat = X_test.copy()
for c in cat_cols:
    X_cat[c] = X_cat[c].astype(str)
    X_test_cat[c] = X_test_cat[c].astype(str)

## Pembagian Data — GroupKFold (perbaikan dari sigma3)

Bedanya dengan `KFold` biasa: `GroupKFold` memastikan semua baris dengan `source_id` yang sama selalu berada di grup yang sama (tidak terpecah ke fold latihan dan fold uji sekaligus). Ini bikin hasil validasi lebih mencerminkan performa asli di data yang benar-benar baru.

In [15]:
gkf = GroupKFold(n_splits=N_FOLDS)
tune_gkf = GroupKFold(n_splits=TUNE_FOLDS)

In [16]:
def lgb_cv_rmse(params):
    oof = np.zeros(len(X))
    for tr_idx, val_idx in tune_gkf.split(X, y, groups):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        dtr = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols)
        dval = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_cols, reference=dtr)
        model = lgb.train(params, dtr, num_boost_round=1500, valid_sets=[dval],
                           callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
        oof[val_idx] = model.predict(X_val, num_iteration=model.best_iteration)
    return mean_squared_error(y, oof) ** 0.5

In [17]:
def lgb_objective(trial):
    params = {
        'objective': 'regression', 'metric': 'rmse', 'verbosity': -1, 'seed': RANDOM_STATE,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 60),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': 1,
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-3, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-3, 10.0, log=True),
    }
    return lgb_cv_rmse(params)

In [18]:
print("Tuning LightGBM...")
lgb_study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
lgb_study.optimize(lgb_objective, n_trials=N_TRIALS, show_progress_bar=False)
print("Best LGB CV RMSE:", lgb_study.best_value)
print("Best LGB params:", lgb_study.best_params)

Tuning LightGBM...
Best LGB CV RMSE: 0.4265646193310858
Best LGB params: {'learning_rate': 0.04779096404331236, 'num_leaves': 102, 'min_data_in_leaf': 55, 'feature_fraction': 0.8725081763159884, 'bagging_fraction': 0.7833294534454811, 'lambda_l1': 0.003770205918821814, 'lambda_l2': 0.016644195020875746}


In [19]:
def xgb_cv_rmse(params):
    oof = np.zeros(len(X_xgb))
    for tr_idx, val_idx in tune_gkf.split(X_xgb, y, groups):
        X_tr, X_val = X_xgb.iloc[tr_idx], X_xgb.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        model = xgb.XGBRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        oof[val_idx] = model.predict(X_val)
    return mean_squared_error(y, oof) ** 0.5

def xgb_objective(trial):
    params = {
        'objective': 'reg:squarederror', 'tree_method': 'hist', 'enable_categorical': False,
        'random_state': RANDOM_STATE, 'n_estimators': 1500, 'early_stopping_rounds': 50,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'n_jobs': -1, 'verbosity': 0,
    }
    return xgb_cv_rmse(params)

In [20]:
print("Tuning XGBoost...")
xgb_study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS, show_progress_bar=False)
print("Best XGB CV RMSE:", xgb_study.best_value)
print("Best XGB params:", xgb_study.best_params)

Tuning XGBoost...
Best XGB CV RMSE: 0.43210029148559026
Best XGB params: {'learning_rate': 0.03410079904079649, 'max_depth': 6, 'min_child_weight': 13, 'subsample': 0.5054063424743919, 'colsample_bytree': 0.5449735679543182, 'reg_alpha': 2.360145095253706, 'reg_lambda': 3.4744592347331045}


## BARU: Tuning CatBoost

CatBoost adalah mesin penebak ketiga yang biasanya kuat kalau banyak kolom kategori (teks) seperti di data ini (jenis tanah, wilayah, dll).

In [21]:
def cat_cv_rmse(params):
    oof = np.zeros(len(X_cat))
    for tr_idx, val_idx in tune_gkf.split(X_cat, y, groups):
        X_tr, X_val = X_cat.iloc[tr_idx], X_cat.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        model = CatBoostRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), cat_features=cat_col_idx,
                  use_best_model=True, verbose=False)
        oof[val_idx] = model.predict(X_val)
    return mean_squared_error(y, oof) ** 0.5

def cat_objective(trial):
    params = {
        'loss_function': 'RMSE', 'random_seed': RANDOM_STATE, 'iterations': 1500,
        'early_stopping_rounds': 100, 'verbose': False,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
    }
    return cat_cv_rmse(params)

In [22]:
print("Tuning CatBoost...")
cat_study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
cat_study.optimize(cat_objective, n_trials=N_TRIALS, show_progress_bar=False)
print("Best CatBoost CV RMSE:", cat_study.best_value)
print("Best CatBoost params:", cat_study.best_params)

Tuning CatBoost...
Best CatBoost CV RMSE: 0.443534046916898
Best CatBoost params: {'learning_rate': 0.08813259224734667, 'depth': 6, 'l2_leaf_reg': 3.6342626543479755, 'bagging_temperature': 0.017961875614819656}


In [23]:
best_lgb_params = {'objective': 'regression', 'metric': 'rmse', 'verbosity': -1, 'seed': RANDOM_STATE}
best_lgb_params.update(lgb_study.best_params)
best_lgb_params['bagging_freq'] = 1

best_xgb_params = {'objective': 'reg:squarederror', 'tree_method': 'hist', 'enable_categorical': False,
                    'random_state': RANDOM_STATE, 'n_estimators': 3000, 'early_stopping_rounds': 100,
                    'n_jobs': -1, 'verbosity': 0}
best_xgb_params.update(xgb_study.best_params)

best_cat_params = {'loss_function': 'RMSE', 'random_seed': RANDOM_STATE, 'iterations': 3000,
                    'early_stopping_rounds': 150, 'verbose': False}
best_cat_params.update(cat_study.best_params)

In [24]:
oof_lgb = np.zeros(len(X)); test_lgb = np.zeros(len(X_test))
oof_xgb = np.zeros(len(X_xgb)); test_xgb_pred = np.zeros(len(X_test_xgb))
oof_cat = np.zeros(len(X_cat)); test_cat_pred = np.zeros(len(X_test_cat))

In [25]:
for tr_idx, val_idx in gkf.split(X, y, groups):
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    # LightGBM
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    dtr = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols)
    dval = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_cols, reference=dtr)
    m_lgb = lgb.train(best_lgb_params, dtr, num_boost_round=1500, valid_sets=[dval],
                       callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
    oof_lgb[val_idx] = m_lgb.predict(X_val, num_iteration=m_lgb.best_iteration)
    test_lgb += m_lgb.predict(X_test, num_iteration=m_lgb.best_iteration) / N_FOLDS

    # XGBoost
    X_tr_x, X_val_x = X_xgb.iloc[tr_idx], X_xgb.iloc[val_idx]
    m_xgb = xgb.XGBRegressor(**best_xgb_params)
    m_xgb.fit(X_tr_x, y_tr, eval_set=[(X_val_x, y_val)], verbose=False)
    oof_xgb[val_idx] = m_xgb.predict(X_val_x)
    test_xgb_pred += m_xgb.predict(X_test_xgb) / N_FOLDS

    # CatBoost (BARU)
    X_tr_c, X_val_c = X_cat.iloc[tr_idx], X_cat.iloc[val_idx]
    m_cat = CatBoostRegressor(**best_cat_params)
    m_cat.fit(X_tr_c, y_tr, eval_set=(X_val_c, y_val), cat_features=cat_col_idx,
              use_best_model=True, verbose=False)
    oof_cat[val_idx] = m_cat.predict(X_val_c)
    test_cat_pred += m_cat.predict(X_test_cat) / N_FOLDS

In [26]:
rmse_lgb = mean_squared_error(y, oof_lgb) ** 0.5
rmse_xgb = mean_squared_error(y, oof_xgb) ** 0.5
rmse_cat = mean_squared_error(y, oof_cat) ** 0.5
print(f"\nFinal OOF RMSE - LightGBM: {rmse_lgb:.5f}")
print(f"Final OOF RMSE - XGBoost : {rmse_xgb:.5f}")
print(f"Final OOF RMSE - CatBoost: {rmse_cat:.5f}")


Final OOF RMSE - LightGBM: 0.43858
Final OOF RMSE - XGBoost : 0.42577
Final OOF RMSE - CatBoost: 0.44477


## Blending 3 model (perbaikan dari sigma3)

Sebelumnya cuma dicoba campuran 2 model dengan 1 angka bobot. Sekarang dicoba kombinasi bobot 3 model sekaligus (grid search sederhana), supaya lebih optimal.

In [27]:
best_weights, best_rmse = (1/3, 1/3, 1/3), 1e9
step = 0.05
for w_lgb in np.arange(0, 1.01, step):
    for w_xgb in np.arange(0, 1.01 - w_lgb, step):
        w_cat = 1 - w_lgb - w_xgb
        blend_oof = w_lgb * oof_lgb + w_xgb * oof_xgb + w_cat * oof_cat
        r = mean_squared_error(y, blend_oof) ** 0.5
        if r < best_rmse:
            best_rmse = r
            best_weights = (w_lgb, w_xgb, w_cat)

In [28]:
w_lgb, w_xgb, w_cat = best_weights
print(f"Best blend weight (LGB={w_lgb:.2f}, XGB={w_xgb:.2f}, CAT={w_cat:.2f}) -> OOF RMSE: {best_rmse:.5f}")

Best blend weight (LGB=0.30, XGB=0.70, CAT=-0.00) -> OOF RMSE: 0.42303


In [29]:
test_blend = w_lgb * test_lgb + w_xgb * test_xgb_pred + w_cat * test_cat_pred
test_blend = np.expm1(test_blend)
submission = pd.DataFrame({ID_COL: test_fe[ID_COL], TARGET: test_blend})
submission.to_csv('submission-sigma4.csv', index=False)
print("Selesai! File tersimpan sebagai submission-sigma4.csv")

Selesai! File tersimpan sebagai submission-sigma4.csv
